In [23]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from dataset_ood_download import get_data_list
import selfies as sf
from augment_dataset import mol2graph

In [2]:
data_tag = "ablation-full-tasks_0509"

tasks = [
    "bace",
    "smol-property_prediction-bbbp",
    "smol-property_prediction-clintox",
    "smol-property_prediction-hiv",
    "smol-property_prediction-sider",
    "smol-property_prediction-esol",
    "smol-property_prediction-lipo",
    "qm9_homo",
    "qm9_lumo",
    "qm9_homo_lumo_gap",
    "forward_reaction_prediction",
    "retrosynthesis",
    "reagent_prediction",
    "chebi-20-text2mol",
    "chebi-20-mol2text",
]

orderly_forward_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly-forward_reaction_prediction'
orderly_retro_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly-retrosynthesis'
orderly_forward_data = datasets.load_from_disk(orderly_forward_path)
orderly_retro_data = datasets.load_from_disk(orderly_retro_path)


molinst_forward_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_forward_reaction_prediction_0219'
molinst_retro_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_retrosynthesis_0219'
smol_forward_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-forward_synthesis_0219'
smol_retro_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-retrosynthesis_0219'
# load datasets
molinst_forward_data = datasets.load_from_disk(molinst_forward_path)
smol_forward_data = datasets.load_from_disk(smol_forward_path)
forward_data = datasets.concatenate_datasets([molinst_forward_data, smol_forward_data])

molinst_retro_data = datasets.load_from_disk(molinst_retro_path)
smol_retro_data = datasets.load_from_disk(smol_retro_path)
retro_data = datasets.concatenate_datasets([molinst_retro_data, smol_retro_data])


In [4]:
data_instance = orderly_forward_data[0]

In [3]:
def get_mol_dict(data_instance):
    input_mol_string = data_instance['input_mol_string'].replace("<SELFIES>", "").replace("</SELFIES>", "").replace(" ", "")
    smiles = sf.decoder(input_mol_string)
# convert to rdkit mol
    smiles = sf.decoder(input_mol_string)
# convert to canonical smiles
    smiles = Chem.MolToSmiles(Chem.MolFromSmiles(smiles), canonical=True)
    mol = Chem.MolFromSmiles(smiles)
# convert to inchikey
    inchikey = Chem.inchi.MolToInchiKey(mol)
    dict = {
    'selfies': input_mol_string,
    'canonical_smiles': smiles,
    'inchikey': inchikey
    }
    return dict

def get_inchikey(data_instance):
    input_mol_string = data_instance['input_mol_string'].replace("<SELFIES>", "").replace("</SELFIES>", "").replace(" ", "")
    smiles = sf.decoder(input_mol_string)
# convert to rdkit mol
    smiles = sf.decoder(input_mol_string)
# convert to canonical smiles
    smiles = Chem.MolToSmiles(Chem.MolFromSmiles(smiles), canonical=True)
    mol = Chem.MolFromSmiles(smiles)
# convert to inchikey
    inchikey = Chem.inchi.MolToInchiKey(mol)
    return inchikey

In [10]:
orderly_forward_input_mol_dict = []
for i in tqdm(range(len(orderly_forward_data))):
    data_instance = orderly_forward_data[i]
    mol_dict = get_mol_dict(data_instance)
    orderly_forward_input_mol_dict.append(mol_dict)

orderly_forward_input_inchikey_list = []
for i in tqdm(range(len(orderly_forward_input_mol_dict))):
    mol_dict = orderly_forward_input_mol_dict[i]
    orderly_forward_input_inchikey_list.append(mol_dict['inchikey'])

dup_forward_data = forward_data.filter(lambda x: get_inchikey(x) in orderly_forward_input_inchikey_list,
                                       num_proc=200)
print("Dup forward data length: ", len(dup_forward_data))

 17%|█▋        | 166/1000 [00:00<00:02, 298.55it/s][12:07:40] WARNING: not removing hydrogen atom without neighbors
[12:07:40] WARNING: not removing hydrogen atom without neighbors
 88%|████████▊ | 877/1000 [00:02<00:00, 306.76it/s][12:07:42] WARNING: not removing hydrogen atom without neighbors
[12:07:42] WARNING: not removing hydrogen atom without neighbors
100%|██████████| 1000/1000 [00:00<00:00, 1395310.71it/s]


Dup forward data length:  696


In [15]:
dup_forward_data_inckey_list = []
for i in tqdm(range(len(dup_forward_data))):
    data_instance = dup_forward_data[i]
    mol_dict = get_mol_dict(data_instance)
    dup_forward_data_inckey_list.append(mol_dict['inchikey'])
dup_forward_data_inckey_list = list(set(dup_forward_data_inckey_list))
print("Dup forward data inchikey list length: ", len(dup_forward_data_inckey_list))

100%|██████████| 696/696 [00:02<00:00, 306.16it/s]

Dup forward data inchikey list length:  628


In [4]:
orderly_retro_input_mol_dict = []
for i in tqdm(range(len(orderly_retro_data))):
    data_instance = orderly_retro_data[i]
    mol_dict = get_mol_dict(data_instance)
    orderly_retro_input_mol_dict.append(mol_dict)

orderly_retro_input_inchikey_list = []
for i in tqdm(range(len(orderly_retro_input_mol_dict))):
    mol_dict = orderly_retro_input_mol_dict[i]
    orderly_retro_input_inchikey_list.append(mol_dict['inchikey'])

dup_retro_data = retro_data.filter(lambda x: get_inchikey(x) in orderly_retro_input_inchikey_list,
                                       num_proc=200)
print("Dup retro data length: ", len(dup_retro_data))

Filter (num_proc=200): 100%|██████████| 1070419/1070419 [01:33<00:00, 11484.00 examples/s]


Dup retro data length:  967


In [16]:
dup_retro_data_inckey_list = []
for i in tqdm(range(len(dup_retro_data))):
    data_instance = dup_retro_data[i]
    mol_dict = get_mol_dict(data_instance)
    dup_retro_data_inckey_list.append(mol_dict['inchikey'])
dup_retro_data_inckey_list = list(set(dup_retro_data_inckey_list))
print("Dup retro data inckey list length: ", len(dup_retro_data_inckey_list))

100%|██████████| 967/967 [00:02<00:00, 404.16it/s]

Dup retro data inckey list length:  775


In [17]:
molinst_forward_test_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_forward_reaction_prediction_0219'
molinst_retro_test_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_retrosynthesis_0219'
smol_forward_test_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-forward_synthesis_0219'
smol_retro_test_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-retrosynthesis_0219'
# load datasets
molinst_forward_test_data = datasets.load_from_disk(molinst_forward_test_path)
smol_forward_test_data = datasets.load_from_disk(smol_forward_test_path)
forward_data = datasets.concatenate_datasets([molinst_forward_test_data, smol_forward_test_data])

molinst_retro_test_data = datasets.load_from_disk(molinst_retro_test_path)
smol_retro_test_data = datasets.load_from_disk(smol_retro_test_path)
retro_data = datasets.concatenate_datasets([molinst_retro_test_data, smol_retro_test_data])

In [25]:
data_instance['target_text']

'<SELFIES>[N][#C][C][=C][C][=C][Branch1][S][N][Branch1][Ring2][C][C][O][C][C][Branch1][C][F][Branch1][C][F][F][C][=C][Ring1][#C][C][Branch1][C][F][Branch1][C][F][F].[O][C][=C][C][=C][Branch1][C][F][C][=C][Ring1][#Branch1]</SELFIES> </s>'

In [28]:
data_instance = forward_data[0]

def get_input_output_num_node(data_instance):
    input_selfies = data_instance['input_mol_string'].replace("<SELFIES>", "").replace("</SELFIES>", "").replace(" ", "")
    input_smiles = sf.decoder(input_selfies)
    input_mol = Chem.MolFromSmiles(input_smiles)
    input_graph = mol2graph(input_mol)
    input_num_nodes = input_graph['num_nodes']

    output_selfies = data_instance['target_text'].split("<SELFIES>")[-1].split("</SELFIES>")[0]
    output_smiles = sf.decoder(output_selfies)
    output_mol = Chem.MolFromSmiles(output_smiles)
    output_graph = mol2graph(output_mol)
    output_num_nodes = output_graph['num_nodes']
    return input_num_nodes, output_num_nodes


In [31]:
# get forward test data mol dict
forward_test_data_input_num_node = []
forward_test_data_output_num_node = []
for i in tqdm(range(len(forward_data))):
    data_instance = forward_data[i]
    input_num_nodes, output_num_nodes = get_input_output_num_node(data_instance)
    forward_test_data_input_num_node.append(input_num_nodes)
    forward_test_data_output_num_node.append(output_num_nodes)

# get retro test data mol dict
retro_test_data_input_num_node = []
retro_test_data_output_num_node = []
for i in tqdm(range(len(retro_data))):
    data_instance = retro_data[i]
    input_num_nodes, output_num_nodes = get_input_output_num_node(data_instance)
    retro_test_data_input_num_node.append(input_num_nodes)
    retro_test_data_output_num_node.append(output_num_nodes)


  0%|          | 0/5062 [00:00<?, ?it/s][12:57:48] WARNING: not removing hydrogen atom without neighbors
[12:57:48] WARNING: not removing hydrogen atom without neighbors
[12:57:48] WARNING: not removing hydrogen atom without neighbors
[12:57:48] WARNING: not removing hydrogen atom without neighbors
  1%|▏         | 74/5062 [00:00<00:20, 244.74it/s][12:57:49] WARNING: not removing hydrogen atom without neighbors
[12:57:49] WARNING: not removing hydrogen atom without neighbors
[12:57:49] WARNING: not removing hydrogen atom without neighbors
[12:57:49] WARNING: not removing hydrogen atom without neighbors
[12:57:49] WARNING: not removing hydrogen atom without neighbors
[12:57:49] WARNING: not removing hydrogen atom without neighbors
[12:57:49] WARNING: not removing hydrogen atom without neighbors
  6%|▌         | 310/5062 [00:01<00:19, 249.55it/s][12:57:49] WARNING: not removing hydrogen atom without neighbors
[12:57:50] WARNING: not removing hydrogen atom without neighbors
  7%|▋        

In [33]:
# print avg num nodes
print("Forward test data input num nodes===============================")
avg_input_num_nodes = np.mean(forward_test_data_input_num_node)
avg_output_num_nodes = np.mean(forward_test_data_output_num_node)
print("Avg input num nodes: ", avg_input_num_nodes)
print("Avg output num nodes: ", avg_output_num_nodes)
# print std num nodes
std_input_num_nodes = np.std(forward_test_data_input_num_node)
std_output_num_nodes = np.std(forward_test_data_output_num_node)
print("Std input num nodes: ", std_input_num_nodes)
print("Std output num nodes: ", std_output_num_nodes)
# print min num nodes
min_input_num_nodes = np.min(forward_test_data_input_num_node)
min_output_num_nodes = np.min(forward_test_data_output_num_node)
print("Min input num nodes: ", min_input_num_nodes)
print("Min output num nodes: ", min_output_num_nodes)
# print max num nodes
max_input_num_nodes = np.max(forward_test_data_input_num_node)
max_output_num_nodes = np.max(forward_test_data_output_num_node)
print("Max input num nodes: ", max_input_num_nodes)
print("Max output num nodes: ", max_output_num_nodes)


print("Retro test data input num nodes===============================")
# print avg num nodes
avg_input_num_nodes = np.mean(retro_test_data_input_num_node)
avg_output_num_nodes = np.mean(retro_test_data_output_num_node)
print("Avg input num nodes: ", avg_input_num_nodes)
print("Avg output num nodes: ", avg_output_num_nodes)
# print std num nodes
std_input_num_nodes = np.std(retro_test_data_input_num_node)
std_output_num_nodes = np.std(retro_test_data_output_num_node)
print("Std input num nodes: ", std_input_num_nodes)
print("Std output num nodes: ", std_output_num_nodes)
# print min num nodes
min_input_num_nodes = np.min(retro_test_data_input_num_node)
min_output_num_nodes = np.min(retro_test_data_output_num_node)
print("Min input num nodes: ", min_input_num_nodes)
print("Min output num nodes: ", min_output_num_nodes)
# print max num nodes
max_input_num_nodes = np.max(retro_test_data_input_num_node)
max_output_num_nodes = np.max(retro_test_data_output_num_node)
print("Max input num nodes: ", max_input_num_nodes)
print("Max output num nodes: ", max_output_num_nodes)

Forward test data input num nodes===============================
Avg input num nodes:  41.98439352034769
Avg output num nodes:  25.20189648360332
Std input num nodes:  20.274332328846327
Std output num nodes:  9.879935705879008
Min input num nodes:  10
Min output num nodes:  7
Max input num nodes:  216
Max output num nodes:  203
Retro test data input num nodes===============================
Avg input num nodes:  25.42280837858805
Avg output num nodes:  28.955391776570984
Std input num nodes:  9.927689272975465
Std output num nodes:  11.243822026702299
Min input num nodes:  5
Min output num nodes:  6
Max input num nodes:  203
Max output num nodes:  205
